# Phase 2: Feature Engineering

Brief overview: building the team-season feature table that all models 
will train on, combining standings, Four Factors, Net Rating, MVP shares, 
All-NBA presence, and playoff-derived features across the 2024-25 and 
2025-26 seasons.

## 1. Imports

In [1]:
import pandas as pd

## 2. Load Cleaned Game Data

**Question:** What does our Phase 1 output look like once loaded fresh 
into this notebook, and is it ready to aggregate?

We load the cleaned game-level dataset produced in Phase 1 
(`01_data_extraction.ipynb`) as the foundation for all features built 
in this notebook.

In [2]:
# read in the cleaned games data
games_combined = pd.read_csv(r"D:\Data Science Projects\nba-26-27-predictor\data\processed\games_clean.csv")

# shape and preview the data
display(games_combined.shape)
display(games_combined.head())

(5258, 30)

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,SEASON_LABEL,SEASON_TYPE_LABEL
0,22024,1610612754,IND,Indiana Pacers,22401189,2025-04-13,IND @ CLE,W,289,126,...,47,61,27,15,10,13,20,8.0,2024-25,Regular Season
1,22024,1610612761,TOR,Toronto Raptors,22401197,2025-04-13,TOR @ SAS,L,240,118,...,30,42,32,8,1,13,23,-7.0,2024-25,Regular Season
2,22024,1610612757,POR,Portland Trail Blazers,22401199,2025-04-13,POR vs. LAL,W,240,109,...,29,56,27,7,6,14,16,28.0,2024-25,Regular Season
3,22024,1610612742,DAL,Dallas Mavericks,22401194,2025-04-13,DAL @ MEM,L,240,97,...,28,38,20,6,3,18,13,-35.0,2024-25,Regular Season
4,22024,1610612739,CLE,Cleveland Cavaliers,22401189,2025-04-13,CLE vs. IND,L,292,118,...,47,65,21,6,9,21,23,-8.0,2024-25,Regular Season


**Takeaway:** 

The dataset consists of 5,258 rows and 30 columns, representing a complete historical record of the 2024-25 and 2025-26 NBA seasons.The row count is the exact mathematical sum of all games played across both seasons, encompassing both regular season and postseason play. A count of 2,460 matches the exact schedule capacity for a standard NBA regular season where 30 teams play 82 games each.

| Season     | Season Type    | Game Count |
| ---------- | -------------- | ---------- |
| 2024-25    | Regular Season | 2,460      |
| 2024-25    | Playoffs       | 168        |
| 2025-26    | Regular Season | 2,460      |
| 2025-26    | Playoffs       | 170        |
| Total Rows |                | 5,258      |


## 3. Team-Season Standings Aggregation

**Question:** What does each team's season look like when reduced to 
one row — wins, losses, and win percentage?

This becomes the base table every other feature in this notebook joins onto.

In [3]:
# filtering games_combined to only Regular Season rows (SEASON_TYPE_LABEL = 'Regular Season')
games_regular_season = games_combined[games_combined['SEASON_TYPE_LABEL'] == 'Regular Season'].copy()

# creating a WIN column from the WL column
games_regular_season['WIN'] = games_regular_season['WL'].apply(lambda x: 1 if x == 'W' else 0)

# grouping SEASON_LABEL and TEAM_ABBREVIATION, aggregating the total wins (sum of WIN), total games played (count), win percentage (wins / games played)
games_regular_season_grouped = games_regular_season.groupby(['SEASON_LABEL', 'TEAM_ABBREVIATION']).agg({
    'WIN': 'sum',
    'GAME_ID': 'count'
}).reset_index()

# calculating win percentage
games_regular_season_grouped['WIN_PERCENTAGE'] = games_regular_season_grouped['WIN'] / games_regular_season_grouped['GAME_ID']

# preview the grouped data (should be 30 teams x 2 seasons = 60 rows)
display(games_regular_season_grouped.shape)
display(games_regular_season_grouped.head())

(60, 5)

,SEASON_LABEL,TEAM_ABBREVIATION,WIN,GAME_ID,WIN_PERCENTAGE
0,2024-25,ATL,40,82,0.487805
1,2024-25,BKN,26,82,0.317073
2,2024-25,BOS,61,82,0.743902
3,2024-25,CHA,19,82,0.231707
4,2024-25,CHI,39,82,0.475610


**Takeaway:** Aggregating regular season games to team-level yields 60 
rows — 30 teams across 2 seasons — confirming complete coverage with no 
missing or duplicated teams. This table now serves as the base for every 
other feature joined in this notebook.

## 4. Opponent Join

**Question:** How do we get each team's own stats and their opponent's 
stats on the same row, which both Net Rating and precise Four Factors 
require?

Each game currently appears as two separate rows (one per team) sharing 
the same GAME_ID. Self-joining on GAME_ID lets us attach the opponent's 
stat line to each team's row.

In [4]:
# create a copy of games_regular_season with columns renamed to have an "OPP_" prefix for the opponent team stats, keeping game_id and team_abbreviation as is for now
games_regular_season_opp = games_regular_season.copy()
games_regular_season_opp = games_regular_season_opp.rename(columns={
    col: f"OPP_{col}" for col in games_regular_season_opp.columns if col not in ['GAME_ID']
})

# merge games_regular_season with this renamed copy, on GAME_ID
games_regular_season_merged = pd.merge(
    games_regular_season, 
    games_regular_season_opp, 
    on='GAME_ID', 
    how='left', 
)

#  dropping rows where a team got merged with itself as its own opponent
games_regular_season_merged = games_regular_season_merged[
    games_regular_season_merged['TEAM_ABBREVIATION'] != games_regular_season_merged['OPP_TEAM_ABBREVIATION']
]

print("Shape of the merged dataframe:")
display(games_regular_season_merged.shape)

Shape of the merged dataframe:


(4920, 61)

## 5. Four Factors

**Question:** Beyond win percentage, what do the underlying shooting, 
turnover, rebounding, and free-throw tendencies tell us about each 
team's style and efficiency?

The Four Factors (eFG%, TOV%, ORB%, FT Rate) are a standard basketball 
analytics framework for isolating what actually wins games, weighted 
roughly in that priority order.

In [5]:
# on games_regular_season, calculate each four factor at the game-row level first (per team, per game)
games_regular_season_merged['eFG%'] = (games_regular_season_merged['FGM'] + 0.5 * games_regular_season_merged['FG3M']) / games_regular_season_merged['FGA']
games_regular_season_merged['TOV%'] = games_regular_season_merged['TOV'] / (games_regular_season_merged['FGA'] + 0.44 * games_regular_season_merged['FTA'] + games_regular_season_merged['TOV'])
games_regular_season_merged['ORB%'] = games_regular_season_merged['OREB'] / (games_regular_season_merged['OREB'] + games_regular_season_merged['OPP_DREB'])
games_regular_season_merged['FT Rate'] = games_regular_season_merged['FTM'] / games_regular_season_merged['FGA']
print("Shape of the merged dataframe with four factors:")
display(games_regular_season_merged.head())

# groupby SEASON_LABEL and TEAM_ABBREVIATION, taking the mean of each four factor across the season
games_4_factors = games_regular_season_merged.groupby(['SEASON_LABEL', 'TEAM_ABBREVIATION']).agg({
    'eFG%': 'mean',
    'TOV%': 'mean',
    'ORB%': 'mean',
    'FT Rate': 'mean'
}).reset_index()
print("Shape of the four factors dataframe:")
display(games_4_factors.head())

# merging this back onto games_regular_season_grouped, joingin on SEASON_LABEL and TEAM_ABBREVIATION
games_regular_season_grouped = pd.merge(
    games_regular_season_grouped, 
    games_4_factors, 
    on=['SEASON_LABEL', 'TEAM_ABBREVIATION'], 
    how='left'
    )
print("Shape of the grouped dataframe with four factors:")
display(games_regular_season_grouped.head())

Shape of the merged dataframe with four factors:


,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,OPP_TOV,OPP_PF,OPP_PLUS_MINUS,OPP_SEASON_LABEL,OPP_SEASON_TYPE_LABEL,OPP_WIN,eFG%,TOV%,ORB%,FT Rate
1,22024,1610612754,IND,Indiana Pacers,22401189,2025-04-13,IND @ CLE,W,289,126,...,21,23,-8.0,2024-25,Regular Season,0,0.445378,0.091756,0.229508,0.168067
3,22024,1610612761,TOR,Toronto Raptors,22401197,2025-04-13,TOR @ SAS,L,240,118,...,12,13,7.0,2024-25,Regular Season,1,0.546392,0.111073,0.226415,0.123711
5,22024,1610612757,POR,Portland Trail Blazers,22401199,2025-04-13,POR vs. LAL,W,240,109,...,20,21,-28.0,2024-25,Regular Season,0,0.515625,0.117410,0.482143,0.104167
7,22024,1610612742,DAL,Dallas Mavericks,22401194,2025-04-13,DAL @ MEM,L,240,97,...,11,15,35.0,2024-25,Regular Season,1,0.465517,0.156958,0.227273,0.183908
8,22024,1610612739,CLE,Cleveland Cavaliers,22401189,2025-04-13,CLE vs. IND,L,292,118,...,13,20,8.0,2024-25,Regular Season,1,0.460870,0.144152,0.276923,0.104348


Shape of the four factors dataframe:


,SEASON_LABEL,TEAM_ABBREVIATION,eFG%,TOV%,ORB%,FT Rate
0,2024-25,ATL,0.547733,0.126258,0.260635,0.198276
1,2024-25,BKN,0.517414,0.127060,0.242355,0.190842
2,2024-25,BOS,0.561858,0.103155,0.255842,0.170975
3,2024-25,CHA,0.503948,0.132114,0.264301,0.175867
4,2024-25,CHI,0.554492,0.121329,0.222886,0.175299


Shape of the grouped dataframe with four factors:


,SEASON_LABEL,TEAM_ABBREVIATION,WIN,GAME_ID,WIN_PERCENTAGE,eFG%,TOV%,ORB%,FT Rate
0,2024-25,ATL,40,82,0.487805,0.547733,0.126258,0.260635,0.198276
1,2024-25,BKN,26,82,0.317073,0.517414,0.127060,0.242355,0.190842
2,2024-25,BOS,61,82,0.743902,0.561858,0.103155,0.255842,0.170975
3,2024-25,CHA,19,82,0.231707,0.503948,0.132114,0.264301,0.175867
4,2024-25,CHI,39,82,0.475610,0.554492,0.121329,0.222886,0.175299


**Takeaway:** Four Factor values fall within expected ranges across all 
teams (eFG% roughly 0.50–0.56, TOV%/ORB%/FT Rate all reasonable 
decimals), confirming the opponent join and calculations are working 
correctly.

## 6. Net Rating

**Question:** How much better or worse is each team than their 
opponents, on a scale that's fair regardless of pace (fast vs. slow 
teams)?

Net Rating is a standard basketball analytics measure — point 
differential per 100 possessions — that's a stronger signal of true 
team strength than raw win percentage alone.

In [6]:
# estimating possessions per game (formula: FGA - OREB + TOV + 0.44 * FTA)
games_regular_season_merged['POSS'] = games_regular_season_merged['FGA'] - games_regular_season_merged['OREB'] + games_regular_season_merged['TOV'] + 0.44 * games_regular_season_merged['FTA']

# calculating per-game Net Rating (formula: (PTS - OPP_PTS) / POSS * 100)
games_regular_season_merged['NET_RATING'] = (games_regular_season_merged['PTS'] - games_regular_season_merged['OPP_PTS']) / games_regular_season_merged['POSS'] * 100

# grouping by SEASON_LABEL and TEAM_ABBREVIATION, taking the mean of NET_RATING across the season
games_net_rating = games_regular_season_merged.groupby(['SEASON_LABEL', 'TEAM_ABBREVIATION']).agg({
    'NET_RATING': 'mean'
}).reset_index()
print("Shape of the net rating dataframe:", games_net_rating.shape)
display(games_net_rating.head())

# merging onto games_regular_season_grouped, joining on SEASON_LABEL and TEAM_ABBREVIATION
games_regular_season_grouped = pd.merge(
    games_regular_season_grouped, 
    games_net_rating,
    on=['SEASON_LABEL', 'TEAM_ABBREVIATION'],
    how='left'
)
print("Shape of the grouped dataframe with four factors and net rating:", games_regular_season_grouped.shape)
display(games_regular_season_grouped.head(10))

Shape of the net rating dataframe: (60, 3)


,SEASON_LABEL,TEAM_ABBREVIATION,NET_RATING
0,2024-25,ATL,-0.958159
1,2024-25,BKN,-7.125251
2,2024-25,BOS,9.397202
3,2024-25,CHA,-9.050762
4,2024-25,CHI,-1.438779


Shape of the grouped dataframe with four factors and net rating: (60, 10)


,SEASON_LABEL,TEAM_ABBREVIATION,WIN,GAME_ID,WIN_PERCENTAGE,eFG%,TOV%,ORB%,FT Rate,NET_RATING
0,2024-25,ATL,40,82,0.487805,0.547733,0.126258,0.260635,0.198276,-0.958159
1,2024-25,BKN,26,82,0.317073,0.517414,0.127060,0.242355,0.190842,-7.125251
2,2024-25,BOS,61,82,0.743902,0.561858,0.103155,0.255842,0.170975,9.397202
3,2024-25,CHA,19,82,0.231707,0.503948,0.132114,0.264301,0.175867,-9.050762
4,2024-25,CHI,39,82,0.475610,0.554492,0.121329,0.222886,0.175299,-1.438779
5,2024-25,CLE,64,82,0.780488,0.578973,0.110769,0.258348,0.189829,9.442753
6,2024-25,DAL,39,82,0.475610,0.550836,0.119703,0.234250,0.205392,-1.107044
7,2024-25,DEN,50,82,0.609756,0.574615,0.119232,0.266557,0.202515,3.758321
8,2024-25,DET,44,82,0.536585,0.548781,0.124178,0.259457,0.194834,1.934339
9,2024-25,GSW,48,82,0.585366,0.537070,0.116519,0.271509,0.189377,3.262096


**Takeaway:** Net Rating values fall within the expected -10 to +10 
range, with top-performing teams (e.g. Boston in 2024-25) landing 
clearly positive and bottom teams clearly negative — confirming the 
possession estimate and point differential calculation are directionally 
correct.

## 7. MVP Shares + All-NBA Team Presence

**Question:** How much individual star power and award recognition does 
each team carry into a season?

MVP voting share and All-NBA selections are strong proxies for 
top-end talent — a factor that raw box score stats alone can 
undercount, since award voting captures reputation, "eye test" 
impact, and team success together.

In [7]:
# load the manually compiled CSVs 
mvp_shares = pd.read_csv(r'D:\Data Science Projects\nba-26-27-predictor\data\raw\mvp_shares.csv')
all_nba = pd.read_csv(r'D:\Data Science Projects\nba-26-27-predictor\data\raw\all_nba.csv')

In [8]:
# aggregating mvp_shares to team-season level - summing mvp_share per SEASON + TEAM in case multiple players from the same team made the MVP voting list
mvp_shares_team = mvp_shares.groupby(['season', 'team']).agg({
    'mvp_share': 'sum'
}).reset_index()

# aggregating all_nba to team-season level - counting how many players per SEASON + TEAM appear in the ALL-NBA list, and separately sum all_nba_share as a weighted version of the same signal
all_nba_team = all_nba.groupby(['season', 'team']).agg(
    ALL_NBA_COUNT=('player_name', 'count'),
    ALL_NBA_SHARE_SUM=('all_nba_share', 'sum')
).reset_index()

# merging mvp_shares_team onto games_regular_season_grouped
games_regular_season_grouped = pd.merge(
    games_regular_season_grouped, 
    mvp_shares_team, 
    left_on=['SEASON_LABEL', 'TEAM_ABBREVIATION'], 
    right_on=['season', 'team'], 
    how='left'
)

# merging all_nba_team onto games_regular_season_grouped
games_regular_season_grouped = pd.merge(
    games_regular_season_grouped, 
    all_nba_team, 
    left_on=['SEASON_LABEL', 'TEAM_ABBREVIATION'], 
    right_on=['season', 'team'], 
    how='left'
)

games_regular_season_grouped = games_regular_season_grouped.drop(columns=['season_x', 'team_x', 'season_y', 'team_y'])

# filling NaN values with 0 - a team with no mvp votes or no all-nba players
games_regular_season_grouped['mvp_share'] = games_regular_season_grouped['mvp_share'].fillna(0)
games_regular_season_grouped['ALL_NBA_COUNT'] = games_regular_season_grouped['ALL_NBA_COUNT'].fillna(0)
games_regular_season_grouped['ALL_NBA_SHARE_SUM'] = games_regular_season_grouped['ALL_NBA_SHARE_SUM'].fillna(0)

print("Shape of the grouped dataframe with four factors, net rating, and MVP/All-NBA data:", games_regular_season_grouped.shape)
display(games_regular_season_grouped.head(10))


Shape of the grouped dataframe with four factors, net rating, and MVP/All-NBA data: (60, 13)


,SEASON_LABEL,TEAM_ABBREVIATION,WIN,GAME_ID,WIN_PERCENTAGE,eFG%,TOV%,ORB%,FT Rate,NET_RATING,mvp_share,ALL_NBA_COUNT,ALL_NBA_SHARE_SUM
0,2024-25,ATL,40,82,0.487805,0.547733,0.126258,0.260635,0.198276,-0.958159,0.000,0.0,0.000
1,2024-25,BKN,26,82,0.317073,0.517414,0.127060,0.242355,0.190842,-7.125251,0.000,0.0,0.000
2,2024-25,BOS,61,82,0.743902,0.561858,0.103155,0.255842,0.170975,9.397202,0.311,1.0,1.000
3,2024-25,CHA,19,82,0.231707,0.503948,0.132114,0.264301,0.175867,-9.050762,0.000,0.0,0.000
4,2024-25,CHI,39,82,0.475610,0.554492,0.121329,0.222886,0.175299,-1.438779,0.000,0.0,0.000
5,2024-25,CLE,64,82,0.780488,0.578973,0.110769,0.258348,0.189829,9.442753,0.075,2.0,1.304
6,2024-25,DAL,39,82,0.475610,0.550836,0.119703,0.234250,0.205392,-1.107044,0.000,0.0,0.000
7,2024-25,DEN,50,82,0.609756,0.574615,0.119232,0.266557,0.202515,3.758321,0.787,1.0,1.000
8,2024-25,DET,44,82,0.536585,0.548781,0.124178,0.259457,0.194834,1.934339,0.012,1.0,0.446
9,2024-25,GSW,48,82,0.585366,0.537070,0.116519,0.271509,0.189377,3.262096,0.002,1.0,0.488
